# Gemma-Titans Training on TPU with Kauldron

This notebook demonstrates the complete pipeline for training and using the **Gemma-Titans** hybrid model on a Google Cloud TPU v5e using the `Kauldron` framework (DeepMind's standard training library).

### Steps included:
1. **Initialization:** Loading base Gemma weights and randomly initializing Titans NLTM modules using `SkipTitans`.
2. **Pre-training (CPT):** Training *only* the Titans memory layers using `kd.optim.partial_updates` to avoid TPU OOM, utilizing a proper dataset.
3. **Save/Load:** Splitting the PyTree to save only the fine-tuned memory weights.
4. **Dialogue Inference:** Running the model and updating the internal memory cache dynamically.

In [1]:
# 0. Environment Setup

# Clone the new kauldron repository
!git clone https://github.com/google-research/kauldron || true
!pip install -q ./kauldron
# Clone the gemma repository if not already present
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository to fix Gemma format issues
!git clone https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog

!pip install -q flax optax seqio





Cloning into 'kauldron'...
remote: Enumerating objects: 9563, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 9563 (delta 28), reused 20 (delta 20), pack-reused 9436 (from 3)
Receiving objects: 100% (9563/9563), 2.83 MiB | 30.22 MiB/s, done.
Resolving deltas: 100% (6881/6881), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 142.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 

In [2]:
# %rm -rf /content/Titans_jax

In [3]:
!git clone https://github.com/andrew-veriga/Titans_jax.git


Cloning into 'Titans_jax'...
remote: Enumerating objects: 287, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 287 (delta 39), reused 67 (delta 36), pack-reused 214 (from 1)
Receiving objects: 100% (287/287), 24.08 MiB | 24.32 MiB/s, done.
Resolving deltas: 100% (169/169), done.
/content/Titans_jax


In [1]:
# Ensure our local modules are in the Python path
import sys
import os
sys.path.append(os.getcwd())
from kauldron import kd
from kauldron.ktyping import Array

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [2]:
import jax
import jax.numpy as jnp
import optax
from kauldron import kd
import numpy as np
import os
import orbax.checkpoint as ocp

# Gemma imports
from gemma import gm

# Our custom Titans integration
import importlib

%cd Titans_jax
import gemma_titans
importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans
from titans_ckpts import SkipTitans
import titans_tree_utils

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

# Prevent JAX from allocating 100% of TPU memory instantly
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Limit XLA to 85% of TPU HBM to leave room for overhead
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"
# Reduce fragmentation and compilation memory spike
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_highest_priority_async_collectives=true --xla_tpu_enable_data_parallel_all_reduce_opt=true --xla_tpu_memory_bound_loop_fusion_limit=1"
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"


/content/Titans_jax
JAX Backend: tpu
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


## 1. Model initialization

We load previously trained titans weights and merge them with original Gemma weights

In [2]:
!rm saved_titans_delta.zip

rm: cannot remove 'saved_titans_delta.zip': No such file or directory


In [3]:
import shutil
import os

def load_titans_weights(load_dir: str):
    load_path = os.path.abspath(load_dir)
    checkpointer = ocp.StandardCheckpointer()

    # Restore directly without a template (returns dictionary of arrays)
    titans_params = checkpointer.restore(load_path)
    return titans_params

merged_params = None
titans_delta_path = "./saved_titans_delta"
titans_zip = "saved_titans_delta.zip"

if os.path.exists(titans_zip):
    print(f"Unpacking {titans_zip}...")
    shutil.unpack_archive(titans_zip, titans_delta_path)

if os.path.exists(titans_delta_path):
    # Load original Gemma 3 1B IT weights
    print("Loading original Gemma weights...")
    original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)

    # Merge loaded Titans weights into original Gemma params
    print("Merging Titans weights...")
    import titans_tree_utils
    loaded_titans_params = load_titans_weights(titans_delta_path)
    merged_params = titans_tree_utils.merge_titans_params(original_params, loaded_titans_params)
    print("✅ Success: Pre-trained Titans weights loaded and merged.")
else:
    print("⚠️ Note: './saved_titans_delta' not found. Titans layers will be randomly initialized.")

⚠️ Note: './saved_titans_delta' not found. Titans layers will be randomly initialized.


2. Pre-training (CPT)

We use `Kauldron` to orchestrate the training.
- **Dataset:** Instead of a python generator, we use `kd.data.py.Elements`.
- **Model Config:** We use the standard `gm.nn.Gemma3_1B.config`.
- **SkipTitans:** Handles loading Gemma weights while keeping Titans randomized.
- **partial_updates:** Ensures XLA only builds backprop graphs for memory parameters to prevent OOM.

In [4]:
# Define the model configuration
gemma_config = gm.nn.Gemma3_1B.config
gemma_config.titans_layer_indices = [11, 15, 23]

# Initialize model
model = Gemma3_1B_Titans(
    config=gemma_config,
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.input",
)

# Create a proper Kauldron dataset pipeline using TFDS
batch_size = 16
max_length = 1024

def format_squad(x):
    # SQuAD structure: context, question, answers (dict with 'text' list)
    answers = x.get('answers', {}).get('text', [])
    answer = answers[0] if len(answers) > 0 else "Unknown"

    # Robustly handle potential bytes from TFDS
    ctx = x["context"].decode('utf-8') if hasattr(x["context"], 'decode') else x["context"]
    q = x["question"].decode('utf-8') if hasattr(x["question"], 'decode') else x["question"]
    ans = answer.decode('utf-8') if hasattr(answer, 'decode') else answer

    # Create 'src' and 'dst' for the Seq2SeqTask
    x['src'] = "Context: " + ctx + "\n\nQuestion: " + q
    x['dst'] = ans
    return x
def format_triviaqa(x):
    # В trivia_qa/rc структура: 'entity_pages' (list) или 'search_results'
    # Берем первый попавшийся контекст из entity_pages
    ctx = ""
    if x['entity_pages']['wiki_context']:
        ctx = x['entity_pages']['wiki_context'][0]

    q = x["question"]
    # Ответ в TriviaQA обычно в x['answer']['value']
    ans = x['answer']['value']

    x['src'] = f"Context: {ctx}\n\nQuestion: {q}"
    x['dst'] = ans
    return x

tokenizer = gm.text.Gemma3Tokenizer()


In [5]:
# 1. Выбираем TyDi QA
def get_train_dataset_tydi_qa():
    return kd.data.py.Tfds(
        name= "trivia_qa/rc",#"trivia_qa/unfiltered.nocontext",#"bot_adversarial_dialogue",#"tydi_qa/goldp", # Более качественные ответы для генерации
        split="train",
        shuffle=True,
        num_epochs=None,
        batch_size=batch_size,
        num_workers=4,
        transforms=[
            # Твой форматтер format_squad сработает и здесь (поля context/question/answers те же)
            format_triviaqa, # format_squad,
            gm.data.Seq2SeqTask(
                in_prompt="src",
                in_response="dst",
                out_input="input",
                out_target="target",
                out_target_mask="loss_mask",
                tokenizer=tokenizer,
                max_length=max_length,
                truncate=True,
            ),
            kd.data.py.Elements(keep=["input", "target", "loss_mask"]),
        ],
    )

# 2. Настраиваем расписание с Warmup
total_steps = 60000
lr_sched = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=5e-5, # Снижаем пик
    warmup_steps=2000, # Даем время на адаптацию Titans
    decay_steps=total_steps - 2000,
    end_value=1e-6
)

# 3. Оптимизатор с Gradient Clipping и AdamW
optimizer_adamw = optax.MultiSteps(
    kd.optim.partial_updates(
        optax.chain(
            optax.clip_by_global_norm(1.0), # Спасает от шума на графике
            optax.adamw(learning_rate=lr_sched, weight_decay=1e-4),
        ),
        mask=kd.optim.select(["memory", "memory_gate"]),
    ),
    every_k_schedule=4,
)

In [6]:
# --- INITIALIZATION LOGIC ---
# We define a simple class because Kauldron expects an object with a .transform() method
class FullParamsInit(kd.ckpts.InitTransform):
    def __init__(self, params):
        self.params = params
    def transform(self, state: kd.train.TrainState) -> kd.train.TrainState:
        return state.replace(params=self.params)


class TPUMemoryMonitor(kd.train.Callback):
    """Колбэк для мониторинга памяти TPU во время обучения."""
    
    def on_step_end(self, step: int, state: kd.train.TrainState, aux: kd.train.Auxiliaries):
        # Выводим статистику каждые 100 шагов (или как тебе удобно)
        if step % 100 == 0:
            print(f"\n--- TPU Memory Monitor (Step {step}) ---")
            for device in jax.devices():
                # memory_stats() работает только на TPU/GPU
                try:
                    stats = device.memory_stats()
                    used_gb = stats['peak_bytes_in_use'] / 1e9
                    total_gb = stats['limit'] / 1e9
                    print(f"Device {device.id} ({device.device_kind}): {used_gb:.2f} / {total_gb:.2f} GB used")
                except (AttributeError, ValueError):
                    # На CPU этот метод может отсутствовать или возвращать ошибку
                    print(f"Device {device.id} ({device.device_kind}): memory_stats not available")


# Configure Trainer
trainer = kd.train.Trainer(
    seed=42,
    workdir=os.path.abspath('./titans_workdir_squad'),
    train_ds=get_train_dataset_tydi_qa(), ##get_train_dataset(),
    model=model,


    # If we have merged_params from Cell 1, use them directly (much faster).
    # Otherwise, use SkipTitans to load Gemma and randomly init Titans.
    init_transform = FullParamsInit(merged_params) if merged_params is not None else SkipTitans(
        wrapped=gm.ckpts.LoadCheckpoint(
            path=gm.ckpts.CheckpointPath.GEMMA3_1B_IT,
        ),
        workdir=os.path.abspath('./SkipTitans_workdir')
    ),
    num_train_steps = total_steps, #(87600 * 10) // 16, #dataset length times to epoches num divide to batch_size
    train_losses={
        "xentropy": kd.losses.SoftmaxCrossEntropyWithIntLabels(
            logits="preds.logits",
            labels="batch.target",
            mask="batch.loss_mask",
        ),
    },
    optimizer = optimizer_adamw,
    # --- PREVENT TPU OOM & REDUCE COMPILATION SPIKE ---
    # # Using optax.MultiSteps for REAL Gradient Accumulation (Optax 0.2.6)
    # optimizer=optax.MultiSteps(
    #     kd.optim.partial_updates(
    #         optax.adam(learning_rate=1e-4),
    #         mask=kd.optim.select(["memory", "memory_gate"]),
    #     ),
    #     every_k_schedule=4,
    # ),
    checkpointer=kd.ckpts.Checkpointer(save_interval_steps=500),

    # Sharding for TPU / TPU Pods
    sharding=kd.sharding.ShardingStrategy(),
    
    callbacks=[
        TPUMemoryMonitor(),
    ],
)

print("Trainer initialized. Starting compilation and training...")


Trainer initialized. Starting compilation and training...


### 2.5 Monitoring with TensorBoard
Launch TensorBoard to monitor training metrics (Loss, Learning Rate) in real-time.

In [18]:
!pip install -U tensorboard-plugin-profile

In [22]:
import jax.profiler
# Запускает сервер на порту 9999
jax.profiler.start_server(9999)

In [19]:
%reload_ext tensorboard


In [ ]:
%tensorboard --logdir /content/Titans_jax/titans_workdir_squad/

In [ ]:
# run the actual training loop:
state, aux = trainer.train()

In [ ]:
import os
# os._exit(0)

## 3. Save / Load Trained Weights
We don't want to save the entire 1B model, just our new memory projections. We utilize the `split_titans_params` utility.

In [ ]:
def save_titans_weights(state: kd.train.TrainState, save_dir: str):
    # 1. Extract only the Titans weights
    _, titans_params = titans_tree_utils.split_titans_params(state.params)

    import shutil
    save_path = os.path.abspath(save_dir)
    if os.path.exists(save_path):
        shutil.rmtree(save_path)

    checkpointer = ocp.StandardCheckpointer()

    # 2. Correct Orbax syntax: save(path, state)
    # Passing titans_params as the positional state saves only them.
    checkpointer.save(save_path, titans_params)
    checkpointer.wait_until_finished()
    print(f"Saved ONLY Titans weights to {save_path}")

def load_titans_weights(load_dir: str):
    load_path = os.path.abspath(load_dir)
    checkpointer = ocp.StandardCheckpointer()

    # Restore directly without a template (returns dictionary of arrays)
    titans_params = checkpointer.restore(load_path)
    return titans_params

# Usage:
save_titans_weights(state, "./saved_titans_delta")
loaded_titans_params = load_titans_weights("./saved_titans_delta")
original_params, _ = titans_tree_utils.split_titans_params(state.params)
state = state.replace(params=titans_tree_utils.merge_titans_params(original_params, loaded_titans_params))

In [ ]:
!ls ./saved_titans_delta


In [ ]:
import os
import zipfile
import shutil
from google.colab import files

checkpoint_dir = "./saved_titans_delta"

# First, let's check if the directory exists
if not os.path.exists(checkpoint_dir):
    print(f"Error: {checkpoint_dir} does not exist!")
else:
    # Try Colab download first
    try:

        # Create a zip file of the checkpoint
        zip_filename = "saved_titans_delta.zip"

        # Remove old zip if exists
        if os.path.exists(zip_filename):
            os.remove(zip_filename)

        # Create zip file
        shutil.make_archive("saved_titans_delta", 'zip', checkpoint_dir)

        # Download the zip file
        files.download(zip_filename)
        print(f"✓ Downloaded {zip_filename} via Colab")

    except ImportError:
        # Not in Colab, create zip and provide manual download instructions
        print("Not running in Google Colab. Creating zip file...")

        zip_filename = "saved_titans_delta.zip"

        # Remove old zip if exists
        if os.path.exists(zip_filename):
            os.remove(zip_filename)

        # Create zip file
        shutil.make_archive("saved_titans_delta", 'zip', checkpoint_dir)

        # Get file size
        file_size = os.path.getsize(zip_filename) / (1024 * 1024)  # Size in MB

        print(f"\n✓ Created {zip_filename} ({file_size:.2f} MB)")
        print(f"📁 Full path: {os.path.abspath(zip_filename)}")
        print("\nTo download, use one of these methods:")
        print("1. Right-click the file in Jupyter file browser and select 'Download'")
        print("2. Use scp/rsync from your terminal:")
        print(f"   scp user@server:{os.path.abspath(zip_filename)} ./")

In [ ]:
repo_url_raw

In [ ]:
# --- Push to GitHub using Personal Access Token (PAT) ---
from google.colab import userdata
import os

print("Pushing weights to GitHub...")

zip_filename = "saved_titans_delta.zip"
if os.path.exists(zip_filename):
    try:
        # 1. Get PAT from Colab Secrets (Key icon on the left)
        # Make sure you created a secret named 'GITHUB_PAT'
        GITHUB_PAT = userdata.get('GITHUB_PAT')

        # 2. Get repository info
        # Extract owner/repo from current git config
        repo_url_raw = !git config --get remote.origin.url
        repo_url = repo_url_raw[0].replace("https://", "").replace("git@github.com:", "").replace(".git", "")

        # 3. Configure git and push
        !git config --global user.email "colab-automated@google.com"
        !git config --global user.name "Colab Bot"

        # Use PAT in URL for authentication
        authenticated_url = f"https://{GITHUB_PAT}@{repo_url}.git"

        !git add {zip_filename}
        !git commit -m "Update trained Titans weights [automated push]" || echo "No changes to commit"
        !git push {authenticated_url} HEAD:main --force

        print("✅ Success: Weights pushed to GitHub.")
    except Exception as e:
        print(f"❌ Error: {e}")
        print("\nHint: Create a 'GITHUB_PAT' in Colab Secrets (Key icon) with 'repo' permissions.")
else:
    print(f"❌ Error: {zip_filename} not found. Save weights first.")

## 4. Dialogue Inference (Test-Time Dynamic Memory)
During inference, the model passes a `cache` containing both the standard Attention KV-cache AND the Titans Neural Memory State. The model automatically updates its internal state.

In [ ]:
importlib.reload(gemma_titans)
# We use Gemma's built-in Sampler to correctly handle positions, attention masks, and cache updates
sampler = gm.text.Sampler(
    model=model,
    params=state.params,
    tokenizer=tokenizer,
)

print("Simulation: User enters a turn...")
prompt = "Context: The Titans are powerful mythological entities.\n\nQuestion: Who are the Titans?"
prompt = "Titans - технология запоминания контекста во время ответа LLM"
# The sampler will automatically prefill the cache, update Titans memory, and generate text
output = sampler.sample(prompt, max_new_tokens=20)

print("\n--- Generated Response ---")
print(output)
print("\nMemory state updated automatically in cache.")

## 5. (Bonus) Fast Loading via `jax.eval_shape`
In a new session, to avoid wasting time and memory initializing random weights before loading, you can create a zero-memory structural template using `jax.eval_shape`.

In [ ]:
import jax
import jax.numpy as jnp
import orbax.checkpoint as ocp
import titans_tree_utils

# 1. Define model config (must match the training config)
eval_config = gm.nn.Gemma3_1B.config
eval_config.titans_layer_indices = [11, 15, 23]
model_eval = Gemma3_1B_Titans(
    config=eval_config,
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.input",
)

# 2. MAGIC: Get a "hollow" template without allocating HBM memory
template_variables = jax.eval_shape(
    model_eval.init,
    jax.random.PRNGKey(0),
    tokens=jnp.ones((1, 1), dtype=jnp.int32)
)

# 3. Extract the Titans portion of the template
_, titans_template = titans_tree_utils.split_titans_params(template_variables['params'])

# 4. Load the saved weights using this lightweight template
checkpointer = ocp.StandardCheckpointer()
state = load_titans_weights(state, "./saved_titans_delta")
# print("Titans weights loaded successfully!")
sampler = gm.text.Sampler(
    model=model_eval,
    params=state.params,
    tokenizer=tokenizer,
)

print("Simulation: User enters a turn...")
prompt = "Context: The Titans are powerful mythological entities.\n\nQuestion: Who are the Titans?"
prompt = "Titans - технология запоминания контекста во время ответа LLM"
# The sampler will automatically prefill the cache, update Titans memory, and generate text
sampler.sample(prompt, max_new_tokens=20)

In [ ]:
sampler.sample(prompt, max_new_tokens=20)